<a href="https://colab.research.google.com/github/heyanugrah/CNNronaldoMessiClassification/blob/main/Decoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# ==========================================
# MASK GENERATION UTILITIES
# ==========================================

def create_self_attention_mask(target_tokens, pad_token_id=0):
    """
    Creates a combined Causal (Look-Ahead) and Padding mask for Decoder Self-Attention.
    Target tokens shape: [batch_size, tgt_seq_len]
    Returns shape: [batch_size, 1, tgt_seq_len, tgt_seq_len]
    """
    batch_size, tgt_seq_len = target_tokens.shape

    # 1. Create padding mask: 1 for real tokens, 0 for pad tokens
    # Shape: [batch_size, 1, 1, tgt_seq_len]
    padding_mask = (target_tokens != pad_token_id).unsqueeze(1).unsqueeze(2)

    # 2. Create causal look-ahead mask: lower triangular matrix of 1s
    # Shape: [tgt_seq_len, tgt_seq_len]
    causal_mask = torch.tril(torch.ones((tgt_seq_len, tgt_seq_len), device=target_tokens.device)).bool()

    # 3. Combine them using bitwise AND
    # (A token is visible ONLY if it's not padding AND it's in the past/present)
    combined_mask = padding_mask & causal_mask

    return combined_mask.int()


def create_cross_attention_mask(encoder_tokens, pad_token_id=0):
    """
    Creates a simple Padding mask for Decoder Cross-Attention based on Encoder inputs.
    Encoder tokens shape: [batch_size, src_seq_len]
    Returns shape: [batch_size, src_seq_len]
    """
    # 1 for real tokens, 0 for pad tokens
    padding_mask = (encoder_tokens !=  pad_token_id).int()
    return padding_mask


# ==========================================
# DECODER MODULE
# ==========================================

class MultiHeadDecoder(nn.Module):

    def __init__(self, dim, no_of_heads, dropout):
        super().__init__()

        assert dim % no_of_heads == 0

        self.dim = dim
        self.no_of_heads = no_of_heads
        self.head_vector_dim = dim // no_of_heads

        print("head_vector_dim:", self.head_vector_dim)

        # Projections for Self-Attention
        self.Wq = nn.Linear(dim, dim)
        self.Wk = nn.Linear(dim, dim)
        self.Wv = nn.Linear(dim, dim)
        self.Wo = nn.Linear(dim, dim)

        # Projections for Cross-Attention
        self.Wq_cross = nn.Linear(dim, dim)
        self.Wk_cross = nn.Linear(dim, dim)
        self.Wv_cross = nn.Linear(dim, dim)
        self.Wo_cross = nn.Linear(dim, dim)

        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)

        self.ffn = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Linear(dim * 4, dim)
        )

        self.norm3 = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)

    # ------------------------------------------
    # CORE GRANULAR HELPERS
    # ------------------------------------------
    def applyFeedForward(self, x):
        return self.ffn(x)

    def applyNormalization(self, norm_layer, x):
        return norm_layer(x)

    def applyLinear(self, linear_layer, x):
        return linear_layer(x)

    def applySoftmax(self, x, dim=-1):
        return F.softmax(x, dim=dim)

    def applyMask(self, scaled_scores, attention_mask):
        """Reshapes 2D padding masks to 4D, leaving 4D causal masks untouched."""
        if attention_mask is not None:
            if attention_mask.dim() == 2:
                mask_reshaped = attention_mask.unsqueeze(1).unsqueeze(2)
            else:
                mask_reshaped = attention_mask
            scaled_scores = scaled_scores.masked_fill(mask_reshaped == 0, -1e9)
        return scaled_scores

    # ------------------------------------------
    # ATTENTION ENGINES
    # ------------------------------------------
    def scaledDotProductAttention(self, Q, K, V, attention_mask):
        """Calculates standard scaled dot-product attention math."""
        # 1. Scores & Scale
        scores = Q @ K.transpose(-2, -1)
        scaled_scores = scores / math.sqrt(self.head_vector_dim)

        # 2. Filter padding or future tokens via separate mask function
        scaled_scores = self.applyMask(scaled_scores, attention_mask)

        # 3. Softmax Distribution & Dropout
        attention_weights = self.applySoftmax(scaled_scores, dim=-1)
        dropout_weights = self.dropout(attention_weights)

        # 4. Context Vector computation
        context_vector = dropout_weights @ V

        return context_vector, K, V

    def computeMultiHeadAttention(self, q_input, k_input, v_input, Wq_layer, Wk_layer, Wv_layer, Wo_layer, mask):
        """Generic unified multi-head processor acting as Self or Cross attention."""
        batch_size = q_input.shape[0]
        q_len = q_input.shape[1]
        kv_len = k_input.shape[1]

        # 1. Linear Projections
        Q = self.applyLinear(Wq_layer, q_input)
        K = self.applyLinear(Wk_layer, k_input)
        V = self.applyLinear(Wv_layer, v_input)

        # 2. Reshape and transpose head dimension to index 1 [batch, heads, seq, head_dim]
        Q = Q.view(batch_size, q_len, self.no_of_heads, self.head_vector_dim).transpose(1, 2)
        K = K.view(batch_size, kv_len, self.no_of_heads, self.head_vector_dim).transpose(1, 2)
        V = V.view(batch_size, kv_len, self.no_of_heads, self.head_vector_dim).transpose(1, 2)

        # 3. Core Attention Call
        context_vector, K_out, V_out = self.scaledDotProductAttention(Q, K, V, mask)

        # 4. Concatenate heads back together
        context_vector = context_vector.transpose(1, 2).contiguous().view(
            batch_size, q_len, self.dim
        )

        # 5. Dense Output Projection
        output = self.applyLinear(Wo_layer, context_vector)

        return output, K_out, V_out

    # ------------------------------------------
    # FORWARD ROUTING PASS
    # ------------------------------------------
    def forward(self, combined_embeddings, encoder_output, attention_mask, cross_attention_mask=None):

        # Block 1: Masked Self-Attention (Look at generated history, hide future)
        self_attn_out, _, _ = self.computeMultiHeadAttention(
            q_input=combined_embeddings,
            k_input=combined_embeddings,
            v_input=combined_embeddings,
            Wq_layer=self.Wq,
            Wk_layer=self.Wk,
            Wv_layer=self.Wv,
            Wo_layer=self.Wo,
            mask=attention_mask
        )
        residual_1 = self_attn_out + combined_embeddings
        norm_1 = self.applyNormalization(self.norm1, residual_1)

        # Block 2: Cross-Attention (Query the encoder's continuous contexts)
        cross_attn_out, K_cross, V_cross = self.computeMultiHeadAttention(
            q_input=norm_1,
            k_input=encoder_output,
            v_input=encoder_output,
            Wq_layer=self.Wq_cross,
            Wk_layer=self.Wk_cross,
            Wv_layer=self.Wv_cross,
            Wo_layer=self.Wo_cross,
            mask=cross_attention_mask
        )
        residual_2 = cross_attn_out + norm_1
        norm_2 = self.applyNormalization(self.norm2, residual_2)

        # Block 3: Position-Wise Feed Forward Network
        ffn_output = self.applyFeedForward(norm_2)
        dropout_output_2 = self.dropout(ffn_output)

        residual_3 = dropout_output_2 + norm_2
        norm_3 = self.applyNormalization(self.norm3, residual_3)

        return norm_3, K_cross, V_cross


# ==========================================
# VERIFICATION RUNTIME SCRIPT
# ==========================================
if __name__ == "__main__":
    torch.manual_seed(42)

    # Dimensions configuration
    batch_size = 2
    src_seq_len = 5
    tgt_seq_len = 4
    dim = 8
    no_of_heads = 2
    dropout = 0.1

    # Simulated vocabulary integer arrays (0 signifies <pad>)
    mock_encoder_tokens = torch.tensor([[1, 2, 3, 4, 5],
                                        [1, 2, 3, 0, 0]])

    mock_decoder_tokens = torch.tensor([[1, 2, 3, 4],
                                        [1, 2, 0, 0]])

    # 1. Mask calculations
    self_attention_mask = create_self_attention_mask(mock_decoder_tokens, pad_token_id=0)
    cross_attention_mask = create_cross_attention_mask(mock_encoder_tokens, pad_token_id=0)

    # 2. Embedding space simulation
    mock_combined_embeddings = torch.randn(batch_size, tgt_seq_len, dim)
    mock_encoder_output = torch.randn(batch_size, src_seq_len, dim)

    # 3. Model construction and calculation
    decoder_layer = MultiHeadDecoder(dim=dim, no_of_heads=no_of_heads, dropout=dropout)

    output, K_cross, V_cross = decoder_layer(
        combined_embeddings=mock_combined_embeddings,
        encoder_output=mock_encoder_output,
        attention_mask=self_attention_mask,
        cross_attention_mask=cross_attention_mask
    )

    print("-" * 50)
    print("Process Finished Successfully!")
    print("Decoder Final Tensor Shape:", output.shape)

head_vector_dim: 4
--------------------------------------------------
Process Finished Successfully!
Decoder Final Tensor Shape: torch.Size([2, 4, 8])
